In [ ]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

# Evaluating Code Given as a String

TypeScript provides the function `eval`, which executes code that is given as a string.  However, a function that is defined by the code passed to `eval` is not available outside of `eval`.  Therefore, the TypeScript version of this notebook defines a function `myEval` that makes these functions globally available.

F$^\sharp$ Interactive has no function `eval`.  However, the *.NET Interactive* kernel that runs this notebook can be asked to execute a string as if it were the content of a new cell.  In F$^\sharp$, every definition of a cell is available in all cells that are executed later.  Therefore, a function that is defined in such a string is available afterwards, too.

The function `myEval` takes one argument:
- `code` is a string containing F$^\sharp$ code.

It sends the *command* `SubmitCode code` to the F$^\sharp$ kernel, which is found by the name `"fsharp"`.  This command is executed after the current cell has finished.  Hence, the definitions in `code` are available in the next cell.  The result of `myEval` is `()`.  Its implementation needs some details of .NET Interactive:
  * The command is sent by `kernel.SendAsync`, which immediately returns a *task*.  The task finishes when the command has been executed.  The function `myEval` does not wait for the task, since the command is only executed after the current cell has finished.
  * If the command were sent from the current cell directly, it would be executed as a part of the current cell and the F$^\sharp$ kernel would forget its definitions once the current cell has finished.  Therefore, the command is sent from a new task that is started by `Task.Run`, and `ExecutionContext.SuppressFlow()` prevents this task from inheriting the context of the current cell.

The function `importNotebook` defined in the script `setup.fsx` uses the same technique.

In [ ]:
open System.Threading
open System.Threading.Tasks
open Microsoft.DotNet.Interactive
open Microsoft.DotNet.Interactive.Commands

let myEval (code: string) : unit =
    let kernel = Kernel.Root.FindKernelByName "fsharp"
    use _ = ExecutionContext.SuppressFlow()
    Task.Run(fun () -> kernel.SendAsync(SubmitCode code) :> Task) |> ignore

The string `t` defines the function `greetJupyter`, which takes a name and returns a greeting.

In [ ]:
let t : string = """
let greetJupyter (name: string) : string =
    "Hello " + name + ", I am globally available!"
"""

In [ ]:
myEval t

After the previous cell has finished, the function `greetJupyter` can be used like any other function.

In [ ]:
greetJupyter "World"